In [ ]:
# Better not to use this since this was used for training the 2 Regression Models one by one on Ammonia and Chlorophyll levels. I have added the generation and prediction
# in the Leaf GAN.ipynb file where you can generate a leaf and predict its values. You can use this file to understand the architectures

In [1]:
from keras.layers import Concatenate
import seaborn as sb
import os
from PIL import Image
from keras.layers import Input
import pandas as pd
import tensorflow as tf
import numpy as np
import cv2
import tensorflow_hub as hub
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report,confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import MaxPooling2D, Flatten, Conv2D, Dense, BatchNormalization, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications.densenet import DenseNet169
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score

In [ ]:
X_Train = []
for image in os.listdir('ProjectBackUp/Images/data'):
    img = Image.open('ProjectBackUp/Images/data/' + image)
    img = img.resize((128, 128))
    X_Train.append(np.array(img) / 255)

In [ ]:
Y_Train = []
df1 = pd.read_csv('ProjectBackUp/CSV/Ammonia.csv', header = None)
df2 = pd.read_csv('ProjectBackUp/CSV/Chlorophyll.csv', header = None)
df1 = df1.to_numpy()
df2 = df2.to_numpy()
for i in range(len(df1)):
    Y_Train.append([df1[i]])

In [ ]:
X_Train = np.array(X_Train)
Y_Train = np.array(Y_Train)
Y_Train = Y_Train.reshape(Y_Train.shape[0], 1, 1)

In [ ]:
X_Train, X_Valid, Y_Train, Y_Valid = train_test_split(X_Train, Y_Train, test_size = 0.1, random_state = 42)

In [ ]:
X_Train.shape, X_Valid.shape, Y_Train.shape, Y_Valid.shape

((1071, 128, 128, 3), (120, 128, 128, 3), (1071, 1, 1), (120, 1, 1))

In [ ]:
in_shape = (128, 128, 3)
in_layer = Input(shape = in_shape)
mobilenet = MobileNetV2(weights = 'imagenet', input_shape = in_shape, include_top = False)
densenet = DenseNet169(weights = 'imagenet', input_shape = in_shape, include_top = False)
for layer in mobilenet.layers:
    layer.trainable = False
for layer in densenet.layers:
    layer.trainable = False
model_mobilenet = mobilenet(in_layer)
model_mobilenet = GlobalAveragePooling2D()(model_mobilenet)
output_mobilenet = Flatten()(model_mobilenet)
model_densenet = densenet(in_layer)
model_densenet = GlobalAveragePooling2D()(model_densenet)
output_densenet = Flatten()(model_densenet)
concatenated = Concatenate()([output_mobilenet, output_densenet])
# X = BatchNormalization()(concatenated)
# X = Dense(256, activation = 'relu')(X)
# X = Dropout(0.5)(X)
# X = BatchNormalization()(concatenated)
X = Dense(1024, activation = 'relu')(concatenated)
X = Dropout(0.5)(X)
X = Dense(1, activation = 'linear')(X)
model = tf.keras.models.Model(inputs = in_layer, outputs = X)

In [ ]:
@tf.function
def RSquare(y, prediction):
    unexplained_error = tf.reduce_sum(tf.square(tf.subtract(y, prediction)))
    total_error = tf.reduce_sum(tf.square(tf.subtract(y, tf.reduce_mean(y))))
    R_squared = tf.subtract(1.0, tf.divide(unexplained_error, total_error))
    return R_squared

In [ ]:
optim = Adam(learning_rate = 0.001)
model.compile(loss = 'mean_squared_error', optimizer = optim, metrics = ['RootMeanSquaredError', RSquare, 'mse'])

In [ ]:
#Earlystopping = EarlyStopping(monitor = 'val_loss', min_delta = 0.01, patience = 6, verbose = 1, mode = 'auto', baseline = None, restore_best_weights = True)
rlr = ReduceLROnPlateau(monitor = "val_loss", factor = 0.01, patience = 6, verbose = 0, mode = "max", min_delta = 0.01)
model_save = ModelCheckpoint('./stacked_model_Ammonia.h5', save_best_only = True, save_weights_only = False, monitor = 'val_loss', mode = 'min', verbose = 1)

In [ ]:
train_samples = 1071
validation_samples = 120
epochs = 20
batch_size = 16

In [ ]:
history = model.fit(X_Train, Y_Train, steps_per_epoch = train_samples // batch_size, epochs = epochs, validation_data = (X_Valid, Y_Valid), callbacks = [rlr, model_save])

Epoch 1/20
66/66 [==============================] - ETA: 0s - loss: 3198.9270 - root_mean_squared_error: 56.5591 - RSquare: -2.2194 - mse: 3198.9270
Epoch 1: val_loss improved from inf to 2023.54114, saving model to .\stacked_model_Ammonia.h5
66/66 [==============================] - 32s 245ms/step - loss: 3198.9270 - root_mean_squared_error: 56.5591 - RSquare: -2.2194 - mse: 3198.9270 - val_loss: 2023.5411 - val_root_mean_squared_error: 44.9838 - val_RSquare: 0.0201 - val_mse: 2023.5411 - lr: 0.0010
Epoch 2/20
66/66 [==============================] - ETA: 0s - loss: 2490.4009 - root_mean_squared_error: 49.9039 - RSquare: -1.2853 - mse: 2490.4009
Epoch 2: val_loss improved from 2023.54114 to 1621.99402, saving model to .\stacked_model_Ammonia.h5
66/66 [==============================] - 10s 154ms/step - loss: 2490.4009 - root_mean_squared_error: 49.9039 - RSquare: -1.2853 - mse: 2490.4009 - val_loss: 1621.9940 - val_root_mean_squared_error: 40.2740 - val_RSquare: 0.2080 - val_mse: 1621.9

Epoch 18/20
66/66 [==============================] - ETA: 0s - loss: 1320.9980 - root_mean_squared_error: 36.3455 - RSquare: -0.1006 - mse: 1320.9980
Epoch 18: val_loss did not improve from 1156.70117
66/66 [==============================] - 9s 130ms/step - loss: 1320.9980 - root_mean_squared_error: 36.3455 - RSquare: -0.1006 - mse: 1320.9980 - val_loss: 1156.8828 - val_root_mean_squared_error: 34.0130 - val_RSquare: 0.3533 - val_mse: 1156.8828 - lr: 1.0000e-07
Epoch 19/20
66/66 [==============================] - ETA: 0s - loss: 1364.9199 - root_mean_squared_error: 36.9448 - RSquare: 0.4724 - mse: 1364.9199
Epoch 19: val_loss did not improve from 1156.70117
66/66 [==============================] - 8s 125ms/step - loss: 1364.9199 - root_mean_squared_error: 36.9448 - RSquare: 0.4724 - mse: 1364.9199 - val_loss: 1156.9039 - val_root_mean_squared_error: 34.0133 - val_RSquare: 0.3533 - val_mse: 1156.9039 - lr: 1.0000e-07
Epoch 20/20
 6/66 [=>............................] - ETA: 7s - loss: 1


Epoch 20: val_loss did not improve from 1156.70117
66/66 [==============================] - 1s 18ms/step - loss: 1084.1042 - root_mean_squared_error: 32.9257 - RSquare: 0.4106 - mse: 1084.1042 - val_loss: 1156.9039 - val_root_mean_squared_error: 34.0133 - val_RSquare: 0.3533 - val_mse: 1156.9039 - lr: 1.0000e-09


# Load the model by any one command depending on the task

# When predicting, load the image and append it to an array and convert it to numpy array. Add another dimension for batch, e.g. 1x128x128x3 for a single image

In [ ]:
# model = tf.keras.models.load_model('stacked_model_Ammonia.h5', custom_objects={"RSquare": RSquare})
# model = tf.keras.models.load_model('stacked_model_Chlorophyll.h5', custom_objects={"RSquare": RSquare})

In [ ]:
Y_Pred = model.predict(X_Test)

4/4 [==============================] - 6s 163ms/step


In [ ]:
for i in range(len(Y_Test)):
    print(Y_Pred[i], Y_Test[i])

[79.6766] [[66.16]]
[-4.190922] [[1.62]]
[25.717487] [[1.64]]
[2.771223] [[1.34]]
[25.182514] [[5.73]]
[6.4043303] [[2.83]]
[26.739925] [[1.14]]
[32.536568] [[20.02]]
[43.14643] [[3.17]]
[94.79618] [[114.07]]
[-4.2800746] [[5.76]]
[18.78714] [[67.]]
[-8.660064] [[1.15]]
[28.333666] [[29.46]]
[43.14816] [[6.01]]
[25.533495] [[10.22]]
[-7.522881] [[2.09]]
[21.443422] [[3.54]]
[62.288483] [[72.76]]
[54.190243] [[43.74]]
[26.662533] [[8.17]]
[29.38946] [[27.63]]
[20.26712] [[19.4]]
[-10.582746] [[7.81]]
[-1.6995282] [[5.43]]
[1.3386608] [[2.93]]
[22.702969] [[26.62]]
[11.745437] [[6.48]]
[-5.8069844] [[5.54]]
[14.005385] [[10.68]]
[24.692421] [[3.35]]
[55.52846] [[105.26]]
[-4.7557955] [[0.9]]
[105.320694] [[114.07]]
[10.457666] [[2.4]]
[102.11188] [[3.18]]
[60.41128] [[10.95]]
[4.7593975] [[7.13]]
[43.795944] [[13.49]]
[-3.5407035] [[7.13]]
[1.3631912] [[6.3]]
[138.86363] [[105.55]]
[-9.22512] [[4.84]]
[9.905407] [[6.06]]
[74.26845] [[27.42]]
[32.93739] [[5.04]]
[37.212776] [[7.47]]
[22.3